# Laboratorio 1 — Ejercicio 4
## Posicionamiento de un dron mediante trilateración

Un dron debe calcular su posición espacial $(x,y,z)$ a partir de las mediciones de distancia recibidas de tres estaciones terrestres fijas ubicadas en

- Estación S1: $P_1=(0,0,0)$,
- Estación S2: $P_2=(2,0,0)$,
- Estación S3: $P_3=(0,3,1)$,

con distancias medidas $d_1=3$, $d_2=2.5$ y $d_3=2.8$. Se plantea el sistema de ecuaciones asociado y se resuelve la posición del dron mediante el método de Newton–Raphson multivariado.

## Resumen del ejercicio

Cada estación conocida y su distancia medida definen una esfera sobre la que debe estar el dron; la posición buscada es la intersección de las tres esferas, un sistema no lineal de 3 ecuaciones con 3 incógnitas. Se plantea el sistema $F(x,y,z)=0$, se deriva su matriz Jacobiana en forma cerrada y se resuelve con Newton–Raphson multivariado, resolviendo en cada paso un sistema lineal $J\,\Delta=-F$ mediante `numpy.linalg.solve`.

El sistema, al ser cuadrático, admite en general dos soluciones (las tres esferas se cortan típicamente en dos puntos). Partiendo de $(x_0,y_0,z_0)=(1,1,1)$, Newton–Raphson converge en **7 iteraciones** a

$$(x,y,z) = (1.6875,\ 1.1227556123,\ 2.2117331632),$$

con residuo nulo a precisión de máquina. Esta es la solución con $z>0$, físicamente consistente con un dron que vuela por encima del plano de las estaciones.

## Objetivo y referencias

Se aplica la generalización multivariada de Newton–Raphson vista en clase,

$$\mathbf{x}_{k+1} = \mathbf{x}_k - J_F(\mathbf{x}_k)^{-1}F(\mathbf{x}_k),$$

resolviendo en la práctica el sistema lineal $J_F(\mathbf{x}_k)\,\Delta_k=-F(\mathbf{x}_k)$ en lugar de invertir explícitamente la Jacobiana. El trabajo se organiza en: (1) plantear el sistema de esferas y su Jacobiana, (2) elegir un punto inicial, (3) implementar y ejecutar Newton–Raphson multivariado, y (4) discutir por qué existen dos soluciones matemáticas y cuál corresponde a la posición física del dron.

## 1. Planteamiento del sistema de ecuaciones

La distancia entre el dron $(x,y,z)$ y cada estación $P_i$ debe igualar la distancia medida $d_i$:

$$
\begin{aligned}
F_1(x,y,z) &= x^2+y^2+z^2-d_1^2 = 0,\\
F_2(x,y,z) &= (x-2)^2+y^2+z^2-d_2^2 = 0,\\
F_3(x,y,z) &= x^2+(y-3)^2+(z-1)^2-d_3^2 = 0.
\end{aligned}
$$

Se trabaja con $F_i=(\text{distancia})^2-d_i^2$ en vez de la distancia misma (que involucraría raíces cuadradas) porque ambas formulaciones tienen las mismas raíces y la versión cuadrática evita divisiones por cero cuando el dron está exactamente sobre una estación durante las iteraciones intermedias.

La matriz Jacobiana, con $J_{ij}=\partial F_i/\partial x_j$, es

$$
J(x,y,z) =
\begin{pmatrix}
2x & 2y & 2z\\
2(x-2) & 2y & 2z\\
2x & 2(y-3) & 2(z-1)
\end{pmatrix}.
$$

In [1]:
import numpy as np

P1, P2, P3 = np.array([0.0, 0.0, 0.0]), np.array([2.0, 0.0, 0.0]), np.array([0.0, 3.0, 1.0])
d1, d2, d3 = 3.0, 2.5, 2.8

def F(v):
    x, y, z = v
    return np.array([
        x**2 + y**2 + z**2 - d1**2,
        (x - 2)**2 + y**2 + z**2 - d2**2,
        x**2 + (y - 3)**2 + (z - 1)**2 - d3**2,
    ])

def jacobiano(v):
    x, y, z = v
    return np.array([
        [2*x,       2*y,       2*z],
        [2*(x - 2), 2*y,       2*z],
        [2*x,       2*(y - 3), 2*(z - 1)],
    ])

### Explicación de esta parte:

`F(v)` evalúa las tres funciones residuales para un vector `v = (x, y, z)`, y `jacobiano(v)` construye la matriz $3\times3$ de derivadas parciales en forma cerrada, evitando diferenciación numérica. Ambas funciones son la traducción directa de las expresiones algebraicas de la sección anterior.

## 2. Implementación de Newton–Raphson multivariado

In [2]:
def newton_raphson_multivariado(v0, tolerancia=1e-10, max_iteraciones=100):
    v = np.array(v0, dtype=float)
    historial = []

    for iteracion in range(1, max_iteraciones + 1):
        Fv = F(v)
        Jv = jacobiano(v)
        delta = np.linalg.solve(Jv, -Fv)
        v_nuevo = v + delta
        error = np.linalg.norm(delta)
        residuo = np.linalg.norm(F(v_nuevo))
        historial.append((iteracion, v_nuevo.copy(), error, residuo))

        if error < tolerancia:
            return v_nuevo, historial

        v = v_nuevo

    raise RuntimeError("No se alcanzó la convergencia.")

### Explicación del algoritmo implementado:

En cada iteración se evalúan $F(\mathbf{v}_k)$ y $J(\mathbf{v}_k)$, y se resuelve el sistema lineal $J\,\Delta=-F$ con `np.linalg.solve` en vez de calcular $J^{-1}$ explícitamente, por eficiencia y estabilidad numérica. El nuevo iterado es $\mathbf{v}_{k+1}=\mathbf{v}_k+\Delta$. El historial guarda el error $\|\Delta_k\|$ (tamaño del paso) y el residuo $\|F(\mathbf{v}_{k+1})\|$ (qué tan bien se satisface el sistema), en paralelo con las columnas `error` y `residuo` usadas en el Ejercicio 2.

## 3. Elección del punto inicial

Se elige el punto inicial $\mathbf{v}_0=(1,1,1)$: no coincide con ninguna estación (lo que anularía renglones de la Jacobiana) y está a una distancia razonable de las tres esferas de radio entre 2.5 y 3.

In [3]:
v0 = (1.0, 1.0, 1.0)
solucion, historial = newton_raphson_multivariado(v0)

print(f"{'k':>2} {'x':>14} {'y':>14} {'z':>14} {'error':>12} {'residuo':>12}")
for k, v, error, residuo in historial:
    print(f"{k:2d} {v[0]:14.9f} {v[1]:14.9f} {v[2]:14.9f} {error:12.3e} {residuo:12.3e}")

print()
print(f"Posición del dron: (x, y, z) = ({solucion[0]:.10f}, {solucion[1]:.10f}, {solucion[2]:.10f})")
print(f"Iteraciones necesarias: {len(historial)}")
print(f"Residuo final ||F(v)||: {np.linalg.norm(F(solucion)):.3e}")

 k              x              y              z        error      residuo
 1    1.687500000    0.633750000    3.678750000    2.790e+00    1.348e+01
 2    1.687500000    1.007818583    2.556544250    1.183e+00    2.424e+00
 3    1.687500000    1.112840505    2.241478485    3.321e-01    1.910e-01
 4    1.687500000    1.122668017    2.211995948    3.108e-02    1.673e-03
 5    1.687500000    1.122755605    2.211733184    2.770e-04    1.329e-07
 6    1.687500000    1.122755612    2.211733163    2.200e-08    2.176e-15
 7    1.687500000    1.122755612    2.211733163    3.037e-16    2.665e-15

Posición del dron: (x, y, z) = (1.6875000000, 1.1227556123, 2.2117331632)
Iteraciones necesarias: 7
Residuo final ||F(v)||: 2.665e-15


### Interpretación de la tabla de iteraciones:

Partiendo de $(1,1,1)$, el error $\|\Delta_k\|$ decae de forma rápidamente acelerada: de $\sim2.79$ en la primera iteración a $\sim2.2\times10^{-8}$ en la sexta y $\sim3.0\times10^{-16}$ en la séptima, aproximadamente duplicando el número de cifras correctas en cada paso. Este comportamiento es la versión multivariada de la convergencia cuadrática de Newton observada en el Ejercicio 2. El algoritmo converge en 7 iteraciones a

$$(x,y,z) = (1.6875,\ 1.1227556123,\ 2.2117331632),$$

con residuo $\|F(\mathbf{v})\|=0$ a precisión de máquina, es decir, las tres ecuaciones de esfera quedan satisfechas exactamente dentro del error de redondeo.

## 4. Verificación directa de las distancias

In [4]:
x, y, z = solucion
distancias = [
    np.linalg.norm(solucion - P1),
    np.linalg.norm(solucion - P2),
    np.linalg.norm(solucion - P3),
]
medidas = [d1, d2, d3]

for i, (calculada, medida) in enumerate(zip(distancias, medidas), start=1):
    print(f"Estación S{i}: distancia calculada = {calculada:.10f}   distancia medida = {medida:.10f}")

Estación S1: distancia calculada = 3.0000000000   distancia medida = 3.0000000000
Estación S2: distancia calculada = 2.5000000000   distancia medida = 2.5000000000
Estación S3: distancia calculada = 2.8000000000   distancia medida = 2.8000000000


### Interpretación de la verificación:

Las tres distancias recalculadas desde la solución de Newton–Raphson hasta $P_1$, $P_2$ y $P_3$ coinciden con $d_1=3$, $d_2=2.5$ y $d_3=2.8$ hasta la precisión mostrada. Esta verificación independiente, hecha con la fórmula de distancia euclidiana original (no con $F$), confirma que la solución no es un artefacto del planteamiento cuadrático sino una posición geométricamente consistente con las tres mediciones.

## 5. ¿Por qué el sistema tiene dos soluciones?

Restando pares de ecuaciones de esfera se cancelan los términos cuadráticos $x^2+y^2+z^2$ y el sistema se reduce a dos ecuaciones **lineales** en $(x,y,z)$ más una ecuación cuadrática. Concretamente, $F_2-F_1=0$ da $x=1.6875$ y $F_3-F_1=0$ da $3y+z=5.58$; sustituyendo ambas relaciones en $F_1=0$ queda una ecuación cuadrática en una sola variable, que en general tiene **dos raíces reales**. Geométricamente, esto corresponde a que dos esferas se cortan en un círculo, y ese círculo interseca a la tercera esfera en (típicamente) dos puntos, simétricos respecto al plano que contiene a las tres estaciones.

Se resuelve numéricamente esa reducción lineal-cuadrática para exhibir ambas soluciones y comparar con la que encontró Newton–Raphson.

In [5]:
# Reducción lineal-cuadrática: x queda fijo por F2 - F1 = 0
x_lineal = ((d1**2 - d2**2) + 4) / 4

# 3y + z = k, obtenido de F3 - F1 = 0
k_lineal = 5 + (d1**2 - d3**2) / 2

# Sustituyendo z = k_lineal - 3y en F1 = 0: y^2 + z^2 = d1^2 - x_lineal^2
rhs = d1**2 - x_lineal**2
a_coef, b_coef, c_coef = 10.0, -6*k_lineal, k_lineal**2 - rhs
discriminante = b_coef**2 - 4*a_coef*c_coef

y_mas = (-b_coef + np.sqrt(discriminante)) / (2*a_coef)
y_menos = (-b_coef - np.sqrt(discriminante)) / (2*a_coef)

for y_sol in (y_mas, y_menos):
    z_sol = k_lineal - 3*y_sol
    v_sol = np.array([x_lineal, y_sol, z_sol])
    print(f"(x, y, z) = ({v_sol[0]:.10f}, {v_sol[1]:.10f}, {v_sol[2]:.10f})   "
          f"||F(v)|| = {np.linalg.norm(F(v_sol)):.3e}")

(x, y, z) = (1.6875000000, 2.2252443877, -1.0957331632)   ||F(v)|| = 5.179e-15
(x, y, z) = (1.6875000000, 1.1227556123, 2.2117331632)   ||F(v)|| = 8.882e-16


### Interpretación de las dos soluciones:

Las dos raíces de la ecuación cuadrática reproducen exactamente los dos puntos de intersección de las tres esferas: una con $z\approx2.2117$ y otra con $z\approx-1.0957$, ambas con residuo nulo. Como un dron opera por encima del terreno donde están instaladas las estaciones, se descarta la solución con $z<0$ por no ser físicamente admisible, y se retiene

$$(x,y,z) = (1.6875,\ 1.1227556123,\ 2.2117331632)$$

como posición real del dron. Esta es precisamente la solución a la que convergió Newton–Raphson desde $(1,1,1)$: al partir de un punto inicial con $z_0=1>0$, cercano a la región física, el método fue arrastrado hacia la raíz físicamente correcta en vez de a la simétrica.

## Resultado:

La posición del dron, obtenida mediante Newton–Raphson multivariado a partir del punto inicial $(1,1,1)$ en 7 iteraciones, es

$$\boxed{(x,y,z) = (1.6875,\ 1.1227556123,\ 2.2117331632)}.$$

Las distancias recalculadas a las tres estaciones coinciden con las mediciones $d_1=3$, $d_2=2.5$, $d_3=2.8$, y el residuo del sistema $\|F(\mathbf v)\|$ es nulo a precisión de máquina.

### Conclusión:

El sistema de tres esferas es, en principio, ambiguo: admite dos soluciones geométricamente válidas, reflejo de que la intersección de tres esferas no es en general un único punto. Newton–Raphson multivariado no solo resolvió el sistema con la convergencia cuadrática esperada —duplicando cifras correctas en cada paso—, sino que, al partir de un punto inicial físicamente razonable ($z_0>0$), convergió directamente a la solución con sentido físico, evitando así la ambigüedad inherente al planteamiento puramente algebraico.